# FIREGUARD ID — 03: Baseline Modeling (K-Means Clustering)
Evaluasi jumlah klaster optimal (k = 2 s.d. 8) menggunakan Elbow Method & Silhouette Score pada data **Train & Validation** (leakage-free), serta baseline 4-tier kerawanan (`k=4`).

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import matplotlib.pyplot as plt
from data_loader import load_splits
from modeling import sweep_kmeans, train_kmeans
from evaluation import evaluate_clustering

train_df, val_df, _ = load_splits('../data/split')
scaled_cols = [c for c in train_df.columns if c.endswith('_scaled')]
X_train = train_df[scaled_cols].values
X_val = val_df[scaled_cols].values

sweep_df = pd.DataFrame(sweep_kmeans(X_train, X_val, k_range=range(2, 9), random_state=42))
sweep_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sweep_df['k'], sweep_df['inertia'], marker='o')
axes[0].set_title('Elbow Method (Inertia)')
axes[0].set_xlabel('k')
axes[1].plot(sweep_df['k'], sweep_df['train_silhouette'], marker='o', label='Train')
axes[1].plot(sweep_df['k'], sweep_df['val_silhouette'], marker='s', label='Validation')
axes[1].set_title('Silhouette Score vs k')
axes[1].set_xlabel('k')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
km4, km4_train_labels = train_kmeans(X_train, n_clusters=4, random_state=42)
km4_val_labels = km4.predict(X_val)
print('Train Metrics (k=4):', evaluate_clustering(X_train, km4_train_labels))
print('Val Metrics   (k=4):', evaluate_clustering(X_val, km4_val_labels))